### Top-$1$ and Top-$k$ Accuracy Test for Classification Models
This notebook performs top-$1$ and top-$k$ accuracy test for classification models that are available at DeGirum model zoo.<br>
* You need to specify your cloud API access token and cloud zoo URL in [env.ini](env.ini)<br>
* You also need to specify the following in the code below: 
  * `MODEL_STR` &mdash; a model identifier string
  * `IMAGES_DIR` &mdash; path to folder containing test images
  * `JSON_FILENAME` &mdash; name of JSON file describing the labels of test images
  * `NUM_CLASSES` &mdash; number of classes in the test dataset
  * `K` &mdash; $k$ in top-$k$<br>
* You can optionally specify the following in the code below:
  * `VERBOSE` &mdash; set print verbosity

#### Specify test options here

In [ ]:
# Specify a classification model from DeGirum model zoo
MODEL_STR = "mobilenet_v2_imagenet--224x224_float_n2x_cpu_1"

# Specify path to the folder containing test images and JSON file
IMAGES_DIR = "imagenet"

# Specify name of the JSON file describing the labels of test images
# Note: JSON file should contain {<img1> : <label1>, <img2> : <label2>, ...} map and be placed inside IMAGES_DIR above
JSON_FILENAME = "image_label_map.json"

# Specify the number of classes in the test dataset 
# For e.g., MNIST and FashionMNIST have 10 classes, ImageNet has 1000 classes
NUM_CLASSES = 1000

# Specify the value of `k` in top-k
K = 5

# Specify verbosity -- set this to True if you wish to see the progress of top-1 and top-k accuracies on the fly
VERBOSE = False

#### Specify where do you want to run your inferences

In [ ]:
import csv
import degirum as dg
import json
import mytools
import os
from random import shuffle

# Get cloud API access token from env.ini file
CLOUD_API_TOKEN = mytools.get_token()

# Get cloud zoo URL from env.ini file
CLOUD_ZOO_URL = mytools.get_cloud_zoo_url()


# Please UNCOMMENT only ONE of the following lines to specify where to run AI inference

# 1. Inference on the DeGirum Cloud Platform
zoo = dg.connect(dg.CLOUD, CLOUD_ZOO_URL, CLOUD_API_TOKEN)

# 2. Inference on DeGirum AI Server deployed on a localhost or on some computer in your LAN or VPN
# zoo = dg.connect(mytools.get_ai_server_hostname(), CLOUD_ZOO_URL, CLOUD_API_TOKEN)

# 3. Inference on DeGirum ORCA accelerator installed on your computer
# zoo = dg.connect(dg.LOCAL, CLOUD_ZOO_URL, CLOUD_API_TOKEN)


#### The rest of the cells below should run without any modifications

In [ ]:
""" 
Load the map from JSON file and image filenames from JSON file or the folder containing subset of images 

"""

if os.path.exists(os.path.join(IMAGES_DIR, JSON_FILENAME)):
    json_file = open(os.path.join(IMAGES_DIR, JSON_FILENAME))
else:
    raise Exception('Error: {} not found!'.format(JSON_FILENAME))
image_label_map = json.load(json_file)
test_images = []
for path, directory, files in os.walk(IMAGES_DIR):
    for file in files:
        if file in image_label_map.keys():
            test_images.append((file, os.path.join(path, file)))
shuffle(test_images)

In [ ]:
""" 
Iterate through the provided list of test images and calculate top-1 and top-k accuracies

"""

# Set print verbosity
verboseprint = print if VERBOSE else lambda *a, **k: None

# Load the model
model = zoo.load_model(MODEL_STR)
num_output_classes = len(model.label_dictionary)

# Set the value of `k` in top-k accuracy computation
if K > 0 and K <= num_output_classes:
    model.output_top_k = int(K)

# Calculate top-1 and top-k accuracies
top_one_accuracy = 0
top_k_accuracy = 0
inference_count = 0
for test_img, test_img_path in test_images:
    category_id = image_label_map[test_img] # O(1) look-up 
    # verboseprint("Inferencing image {}, {}".format(inference_count, test_img_path))
    result = model(test_img_path).results
    if len(result) > 0:
        max_category_id = None
        max_score = -10e5
        category_ids = [res_obj['category_id'] + NUM_CLASSES - num_output_classes for res_obj in result]
        max_category_id = result[0]['category_id'] + NUM_CLASSES - num_output_classes
        if category_id == max_category_id:
            top_one_accuracy += 1
            top_k_accuracy += 1
        elif category_id in category_ids:
            top_k_accuracy += 1
        inference_count += 1
        if inference_count % 1000 == 0:
            verboseprint("Inferencing image {}".format(inference_count))
            verboseprint("True class: {}, Predicted class: {}".format(category_id, max_category_id))
            verboseprint("Top-1 accuracy: {}, Top-{} accuracy: {}".format(top_one_accuracy * 100 / inference_count, model.output_top_k, top_k_accuracy * 100 / inference_count))
            verboseprint("\n")
print("Top-1 accuracy of {} is {}".format(MODEL_STR, top_one_accuracy * 100 / inference_count))
print("Top-{} accuracy of {} is {}".format(model.output_top_k, MODEL_STR, top_k_accuracy * 100 / inference_count))

In [ ]:
"""
Write output to CSV file

"""
if not os.path.exists('topaccuracy'):
    os.mkdir('topaccuracy')
csv_file_path = os.path.join('topaccuracy', 'topacc_class_models.csv')
if not os.path.isfile(csv_file_path):
    with open(csv_file_path, 'x') as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(['model', 'top-1_acc', 'top-'+str(model.output_top_k)+'_acc'])
        writer.writerow([MODEL_STR, top_one_accuracy * 100 / inference_count, top_k_accuracy * 100 / inference_count])
else:
    with open(csv_file_path, 'a') as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow([MODEL_STR, top_one_accuracy * 100 / inference_count, top_k_accuracy * 100 / inference_count])
        